# TabPFN parameters, for a gradient-boosting practitioner

With a gradient booster you spend real effort searching hyperparameters: tree depth, learning rate,
number of trees, subsample, column sample, regularization. With TabPFN that search is gone. The
model is **pretrained**, its weights are fixed, and there is no train-time fit to tune. What is left
is a short list of **inference settings**, and the defaults are strong.

This notebook tours them, measures the few that actually move the needle, and shows how to inspect
what TabPFN decided about your data. Everything runs on a small synthetic task so it stays
self-contained.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from tabpfn import TabPFNClassifier

def make_data(n=6000, n_features=12, noise=1.5, positive_rate=None, seed=0):
    """A synthetic binary task with a nonlinear signal plus noise."""
    rng = np.random.RandomState(seed)
    X = rng.randn(n, n_features)
    signal = X[:, 0] * X[:, 1] + X[:, 2] ** 2 - X[:, 3] * X[:, 4] + 0.8 * X[:, 5]
    logit = signal + noise * rng.randn(n)
    if positive_rate is None:
        label = (logit > np.median(logit)).astype(int)               # balanced
    else:
        label = (logit > np.quantile(logit, 1 - positive_rate)).astype(int)
    table = pd.DataFrame(X, columns=[f"x{i}" for i in range(n_features)])
    return table, label

def expected_calibration_error(y_true, p, bins=15):
    """Average gap between predicted confidence and actual frequency, over equal-width bins."""
    bucket = np.clip(np.digitize(p, np.linspace(0, 1, bins + 1)) - 1, 0, bins - 1)
    error = 0.0
    for b in range(bins):
        in_bin = bucket == b
        if in_bin.any():
            error += in_bin.mean() * abs(y_true[in_bin].mean() - p[in_bin].mean())
    return error

## The map

| Parameter (default) | What it controls | When to touch it |
|---|---|---|
| `n_estimators` (8) | how many differently-preprocessed copies of the data are ensembled | lower for speed, higher for a small accuracy bump |
| `categorical_features_indices` (None) | which columns are treated as categorical | when auto-detection mis-types a column |
| `ignore_pretraining_limits` (False) | allow data beyond the recommended size | many rows or features (expect some degradation) |
| `balance_probabilities` (False) | reweight outputs toward balanced classes | imbalanced data, to move the 0.5 operating point |
| `softmax_temperature` (0.9) | sharpen or soften the predicted probabilities | calibration |
| `inference_config` (None) | override the internal preprocessing | advanced: scaling, outlier clip, categorical thresholds |
| `device`, `inference_precision`, `memory_saving_mode`, `fit_mode` | hardware and memory | performance |
| `model_path` (`'auto'`) | which checkpoint to load | `'auto'` loads v3; rarely changed |
| `tuning_config` (None) | optional post-hoc tuning search | when you want an AutoML-style sweep |
| `random_state` (0) | reproducibility | set it for reproducible runs |

The next cells measure the three a practitioner actually reaches for, then inspect the rest.

## `n_estimators`: the one real accuracy-vs-speed dial

Each estimator is a differently-preprocessed copy of the data (a different scaling and feature
order), and TabPFN averages over them. More estimators means more diversity, and more compute. We
sweep the count and time the prediction.

In [2]:
X, y = make_data()
half = len(X) // 2
X_train, X_test = X.iloc[:half], X.iloc[half:]
y_train, y_test = y[:half], y[half:]

# Warm up once so the first timing is not dominated by one-time GPU kernel setup.
TabPFNClassifier(n_estimators=2, random_state=0).fit(X_train, y_train).predict_proba(X_test)

rows = []
for k in [1, 2, 4, 8, 16, 32]:
    model = TabPFNClassifier(n_estimators=k, random_state=0)
    model.fit(X_train, y_train)
    start = time.time()
    p = model.predict_proba(X_test)[:, 1]
    seconds = time.time() - start
    rows.append({"n_estimators": k,
                 "AUC": round(roc_auc_score(y_test, p), 4),
                 "predict_seconds": round(seconds, 2)})

pd.DataFrame(rows).set_index("n_estimators")

,AUC,predict_seconds
n_estimators,,
1,0.8510,0.28
2,0.8540,0.53
4,0.8542,1.04
8,0.8546,2.05
16,0.8545,4.10
32,0.8548,8.22


AUC climbs from one estimator to a handful and then **flattens**, while predict time grows roughly
**linearly** with the count. So the default of 8 is conservative: you can drop to 2 to 4 for a large
speed-up at a negligible accuracy cost, or raise it to 16 to 32 for a marginal gain. This is the one
setting where there is a real trade-off to make. (Absolute times depend on your hardware.)

## The calibration knobs: ranking versus operating point

`balance_probabilities` and `softmax_temperature` both reshape the output probabilities. The
question a practitioner should ask: do they change the *ranking* (AUC, AUPRC), or only *where* you
threshold and how calibrated the numbers are? We test on an imbalanced task (about 10% positive).

In [3]:
X, y = make_data(positive_rate=0.10, seed=1)          # imbalanced: ~10% positive
half = len(X) // 2
X_train, X_test = X.iloc[:half], X.iloc[half:]
y_train, y_test = np.array(y[:half]), np.array(y[half:])

rows = []
for balance in [False, True]:
    model = TabPFNClassifier(balance_probabilities=balance, random_state=0)
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    predicted_positive = (p >= 0.5).astype(int)
    recall_at_half = predicted_positive[y_test == 1].mean()
    rows.append({"balance_probabilities": balance,
                 "AUC": round(roc_auc_score(y_test, p), 4),
                 "AUPRC": round(average_precision_score(y_test, p), 4),
                 "ECE": round(expected_calibration_error(y_test, p), 4),
                 "recall@0.5": round(recall_at_half, 3)})

pd.DataFrame(rows).set_index("balance_probabilities")

,AUC,AUPRC,ECE,recall@0.5
balance_probabilities,,,,
False,0.9294,0.7027,0.0185,0.446
True,0.9294,0.7027,0.1292,0.812


In [4]:
# softmax_temperature on the balanced task: sharper (<1) or softer (>1) probabilities.
X, y = make_data()
half = len(X) // 2
X_train, X_test = X.iloc[:half], X.iloc[half:]
y_train, y_test = np.array(y[:half]), np.array(y[half:])

rows = []
for temperature in [0.5, 0.9, 2.0]:
    model = TabPFNClassifier(softmax_temperature=temperature, random_state=0)
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    rows.append({"softmax_temperature": temperature,
                 "AUC": round(roc_auc_score(y_test, p), 4),
                 "ECE": round(expected_calibration_error(y_test, p), 4)})

pd.DataFrame(rows).set_index("softmax_temperature")

,AUC,ECE
softmax_temperature,,
0.5,0.8545,0.0998
0.9,0.8546,0.0294
2.0,0.8546,0.0971


Neither knob changes the ranking: AUC (and AUPRC) are identical across `balance_probabilities`, and
flat across `softmax_temperature`. What moves is the **operating point** and the **calibration**.
Balancing pushes far more predictions over 0.5, so `recall@0.5` jumps, but the probabilities drift
away from the true base rate, so ECE gets worse. Temperature only trades sharpness for softness, and
the default 0.9 is already near the best-calibrated point.

The lesson for a gradient-boosting practitioner: unlike `scale_pos_weight`, these do not improve a
ranking metric. Reach for them only if you threshold at 0.5 or need calibrated probabilities.

## Inspect what TabPFN decided

You do not tune the preprocessing, but you can read it. After a fit, `inference_config_` holds the
resolved limits and thresholds, and `inferred_feature_schema_` shows the modality TabPFN assigned to
each column (which is what `categorical_features_indices` would override).

In [ ]:
model = TabPFNClassifier(random_state=0).fit(X_train.head(200), y_train[:200])

print("Envelope (hard ceilings):")
for field in ["MAX_NUMBER_OF_SAMPLES", "MAX_NUMBER_OF_FEATURES", "MAX_NUMBER_OF_CLASSES"]:
    print(f"  {field} = {getattr(model.inference_config_, field)}")

print("\nPreprocessing thresholds:")
for field in ["MAX_UNIQUE_FOR_CATEGORICAL_FEATURES", "MIN_UNIQUE_FOR_NUMERICAL_FEATURES",
              "MIN_NUMBER_SAMPLES_FOR_CATEGORICAL_INFERENCE", "OUTLIER_REMOVAL_STD"]:
    print(f"  {field} = {getattr(model.inference_config_, field)}")

print("\nInferred column modality (first five columns):")
for feature in model.inferred_feature_schema_.features[:5]:
    print(f"  {feature.name}: {feature.modality.value}")

The **envelope** is the hard ceiling the checkpoint was built for (a million rows, two thousand
features, a hundred and sixty classes). Separately, `ignore_pretraining_limits` guards a softer
*recommended* operating range, and pushing well past it is where accuracy starts to slip. The
**thresholds** are the rules behind the auto-detection: a string column with 30 or fewer distinct
values is treated as categorical, a numeric column needs at least 4 distinct values to stay numeric,
and category inference needs at least 100 rows. Those are the knobs the scaling and categorical
chapters open up through `inference_config`.

## Takeaways

- TabPFN is pretrained, so there is no hyperparameter *search*. You configure inference, and the
  defaults are strong.
- **`n_estimators`** is the one real accuracy-versus-speed dial, and even it saturates fast: past a
  handful, AUC barely moves while predict time grows linearly. Drop it for speed, raise it for a
  marginal gain.
- **`balance_probabilities`** and **`softmax_temperature`** never change the ranking. They shift the
  operating point and the calibration, so use them only if you threshold at 0.5 or need calibrated
  probabilities.
- **`categorical_features_indices`** and **`ignore_pretraining_limits`** are about telling TabPFN the
  truth about your data: which columns are categorical, and whether you are pushing past its size.
- **`inference_config`** is the door to the internal preprocessing (scaling, outlier clipping,
  categorical thresholds), explored in the other chapters.

The shift from gradient boosting: you are not tuning a model, you are configuring a fixed one, and
most of the time the defaults are the right answer.